# Stock Analysis Report Generator Using Grok

<small>

#### **Overview**
This notebook automatically generates:

- Individual stock analysis reports with key financial and growth highlights followed by BuccoCapital 13 point analysis framework
- ETF and Mutual Fund analysis with key financial and growth highlights, composition, fees, liquidity, etc in the style of Brian Belsky
- Analyst ratings and price targets (when available)

This notebook uses xAI's **Grok** model via API (OpenAI-compatible). Each report is saved as a formatted Microsoft Word document.

#### **Features**
- Batch Processing: Processes multiple stocks, ETFs and mutual funds from a portfolio CSV or plain text ticker file
- Smart Skipping: Avoids duplicate API calls by checking for existing reports
- Markdown Formatting: Converts AI-generated markdown to professional Word formatting
- Comprehensive Analysis: Includes financial metrics and analyst ratings
- Date Stamping: Automatically timestamps each report

#### **Requirements**
- xAI API Key (set `XAI_API_KEY` in your .env)
- Input: Fidelity portfolio CSV (`PORTFOLIO_CSV_FILE`) or plain text ticker file (`EQUITY_LIST_FILE`)
- Python 3.7+
- Folder structure with inputs and outputs configured in .env

#### **Output**
- Analysis Word document for each equity, ETF, and mutual fund

</small>


## Setup (Run Once)
Only run this cell if packages are not already installed.
The `requirements.txt` file lives in `Input_Source_Files/`.

In [ ]:
# Run this cell ONCE to install dependencies.
# After installation, you do not need to run it again.
# To install: uncomment the line below and run manually in Jupyter.

# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
#     "Input_Source_Files/requirements.txt"])


## Import Libraries

In [ ]:
# ── Core stdlib ──────────────────────────────────────────────────────────
import os
import re
import json
import time
from datetime import datetime

# ── Third-party ───────────────────────────────────────────────────────────
import requests
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Environment ───────────────────────────────────────────────────────────
from dotenv import load_dotenv

# ── LLM — imported here once; GrokClient cell uses these directly ─────────
from openai import OpenAI, RateLimitError

# ── Word document generation ──────────────────────────────────────────────
from docx import Document
from docx.shared import Pt, RGBColor

# NOTE: Cells must be run top-to-bottom. Later cells depend on imports
# and variables defined here and in the Configuration cell.


## Configuration Settings

In [ ]:
# Load environment variables
load_dotenv()

# xAI Grok API key
API_KEY = os.getenv('XAI_API_KEY')

# Input/Output Configuration
INPUT_DIR                           = os.getenv('Input_dir')
OUTPUT_DIR_SEC_FILINGS              = os.getenv('Output_dir_sec_filings')
OUTPUT_DIR_INDIVIDUAL_STOCK_ANALYSIS= os.getenv('Output_dir_individual_equities')
OUTPUT_DIR_PORTFOLIO_ANALYSIS       = os.getenv('Output_dir_portfolio')

# Prompts and Input Lists
EQUITY_LIST_FILE               = os.getenv('EQUITY_LIST_FILE')
PROMPT_DIR                     = os.getenv('Prompt_dir')
PROMPT_INDIVIDUAL_EQUITY_TEMPLATE  = os.getenv('PROMPT_INDIVIDUAL_EQUITY_FILE')
PROMPT_PORTFOLIO_FILE          = os.getenv('PROMPT_PORTFOLIO_FILE')
PROMPT_RATINGS_CHANGE_FILE     = os.getenv('PROMPT_RATINGS_CHANGE_FILE')
PROMPT_ETF_ANALYSIS_FILE       = os.getenv('PROMPT_ETF_ANALYSIS_FILE')

# Model Configuration — xAI Grok (OpenAI-compatible API)
# Override via env: GROK_MODEL, GROK_MAX_TOKENS, GROK_TEMPERATURE
MODEL       = os.getenv('GROK_MODEL',       'grok-3')      # use grok-3 for best analysis quality
TEMPERATURE = int(os.getenv('GROK_TEMPERATURE', '0'))      # 0 = deterministic, factual
MAX_TOKENS  = int(os.getenv('GROK_MAX_TOKENS',  '8192'))   # raised from 2000; grok-3 supports 8192+

# SEC EDGAR header (required by SEC for all requests)
SEC_HEADER = os.getenv('User_Agent')



## Read in Text Inputs

In [ ]:
# ============================================================
# Load Equity List from Fidelity Portfolio CSV
# ============================================================
# Supports both a plain text ticker file (EQUITY_LIST_FILE) and a
# Fidelity CSV export (PORTFOLIO_CSV_FILE).  The CSV path takes priority
# when set.  Filters out cash, money-market, Treasury CUSIPs, options,
# and any rows with blank / placeholder symbols.

# Symbols that are never real tickers in a Fidelity export
_SKIP_SYMBOL_PATTERNS = [
    r'^$',                          # blank
    r'\*\*$',                       # FCASH**, SPAXX**, FDRXX**, CORE**
    r'^[A-Z0-9]{9}$',               # 9-char CUSIP (US Treasuries, e.g. 91282CGS4)
    r'^-[A-Z]+\d',                 # Options (leading dash, e.g. -IOT260717C33)
    r'^Pending',                    # 'Pending activity' rows
]
_SKIP_RE = re.compile('|'.join(_SKIP_SYMBOL_PATTERNS))

# Auto-detect the latest Fidelity_Portfolio_Position*.csv using date in filename
def _find_latest_portfolio_csv():
    """
    Scan Input_Source_Files/ for files matching Fidelity_Portfolio_Position*.csv.
    Extracts the date from the filename and returns the most recent one.
    Supports date formats: Apr-06-2026, April-06-2026, Mar-25-2026, March-25-2026, etc.
    Falls back to file modification time if date cannot be parsed.
    """
    import re as _re
    from datetime import datetime as _dt

    _MONTH_MAP = {
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12,
        'january': 1, 'february': 2, 'march': 3, 'april': 4, 'june': 6,
        'july': 7, 'august': 8, 'september': 9, 'october': 10,
        'november': 11, 'december': 12,
    }

    def _parse_date_from_filename(fname):
        # Match patterns like Apr-06-2026, April-06-2026, Mar-25-2026
        m = _re.search(r'([A-Za-z]+)-(\d{1,2})-(\d{4})', fname)
        if m:
            month_str, day, year = m.group(1).lower(), int(m.group(2)), int(m.group(3))
            month = _MONTH_MAP.get(month_str)
            if month:
                try:
                    return _dt(year, month, day)
                except ValueError:
                    pass
        return None

    input_dir = os.path.join(os.path.dirname(os.path.abspath('portfolio-analysis-notebook.ipynb')),
                             'Input_Source_Files')
    if not os.path.isdir(input_dir):
        return None

    candidates = [
        os.path.join(input_dir, f) for f in os.listdir(input_dir)
        if f.startswith('Fidelity_Portfolio_Position') and f.endswith('.csv')
    ]
    if not candidates:
        return None

    def _sort_key(path):
        date = _parse_date_from_filename(os.path.basename(path))
        return date if date else _dt.fromtimestamp(os.path.getmtime(path))

    return sorted(candidates, key=_sort_key, reverse=True)[0]

PORTFOLIO_CSV_FILE = _find_latest_portfolio_csv()

def load_equity_list_from_csv(csv_path):
    """
    Read a Fidelity portfolio CSV export and return a deduplicated list
    of investable ticker symbols (equities, ETFs, mutual funds).
    Automatically skips cash, money-market, Treasuries, and options.
    """
    df = pd.read_csv(csv_path, thousands=',', encoding='utf-8-sig')
    # Fidelity CSV column is 'Symbol'; strip whitespace
    df['Symbol'] = df['Symbol'].astype(str).str.strip()
    # Drop rows whose symbol matches any skip pattern
    mask = ~df['Symbol'].fillna('').apply(lambda s: bool(_SKIP_RE.search(str(s))))
    tickers = df.loc[mask, 'Symbol'].dropna().unique().tolist()
    # Remove any remaining whitespace or empty strings
    tickers = [t.strip() for t in tickers if t.strip()]
    return tickers

# Determine source: CSV takes priority over plain text file
if PORTFOLIO_CSV_FILE and os.path.exists(PORTFOLIO_CSV_FILE):
    EQUITY_LIST = load_equity_list_from_csv(PORTFOLIO_CSV_FILE)
elif EQUITY_LIST_FILE and os.path.exists(EQUITY_LIST_FILE):
    with open(EQUITY_LIST_FILE, 'r') as f:
        EQUITY_LIST = [line.strip() for line in f if line.strip()]
else:
    raise FileNotFoundError(
        'No equity list found. Set PORTFOLIO_CSV_FILE or EQUITY_LIST_FILE in your .env'
    )

# ── Load prompt templates from files ─────────────────────────────────────
def _load_prompt(path_var, var_name):
    """Load a prompt file; warn and return empty string if missing."""
    path = path_var
    if path and os.path.exists(path):
        with open(path, 'r') as f:
            return f.read().strip()
    return ''

PROMPT_INDIVIDUAL_EQUITY_TEMPLATE   = _load_prompt(PROMPT_INDIVIDUAL_EQUITY_TEMPLATE, 'PROMPT_INDIVIDUAL_EQUITY_TEMPLATE')
PROMPT_ETF_ANALYSIS_FILE_TEMPLATE   = _load_prompt(PROMPT_ETF_ANALYSIS_FILE,         'PROMPT_ETF_ANALYSIS_FILE_TEMPLATE')
PROMPT_PORTFOLIO_TEMPLATE           = _load_prompt(PROMPT_PORTFOLIO_FILE,             'PROMPT_PORTFOLIO_TEMPLATE')
PROMPT_RATINGS_CHANGE_TEMPLATE      = _load_prompt(PROMPT_RATINGS_CHANGE_FILE,        'PROMPT_RATINGS_CHANGE_TEMPLATE')



## Define Grok LLM Client

In [ ]:
# time, OpenAI, and RateLimitError are all imported in the Imports cell.

# ── Grok API rate limits (xAI published limits for grok-3) ───────────────
# Requests per minute : 60  →  1 request per second minimum
# Tokens per minute   : 150,000
# We target ~50 req/min (1.2s between calls) to stay safely under the limit.
GROK_MIN_CALL_INTERVAL = float(os.getenv('GROK_MIN_CALL_INTERVAL', '1.2'))  # seconds
GROK_MAX_RETRIES       = int(os.getenv('GROK_MAX_RETRIES', '5'))

class GrokClient:
    """
    Lightweight wrapper around xAI's Grok API (OpenAI-compatible).

    Rate limiting:
      - Enforces a minimum interval between consecutive calls so the
        notebook stays under xAI's 60 req/min limit for grok-3.
      - On HTTP 429 (rate limit exceeded), backs off exponentially
        and retries up to GROK_MAX_RETRIES times.

    Returns (text, usage_dict) so token usage can be tracked per call.
    """

    def __init__(self, api_key):
        self.client         = OpenAI(api_key=api_key, base_url='https://api.x.ai/v1')
        self._last_call_ts  = 0.0   # timestamp of the last successful call

    def _wait_for_rate_limit(self):
        """Sleep if needed to honour GROK_MIN_CALL_INTERVAL."""
        elapsed = time.monotonic() - self._last_call_ts
        if elapsed < GROK_MIN_CALL_INTERVAL:
            time.sleep(GROK_MIN_CALL_INTERVAL - elapsed)

    def chat(self, message, model=None, max_tokens=None, temperature=None):
        """
        Send a message to Grok and return (text, usage).

        usage dict keys: model, input_tokens, output_tokens, total_tokens, elapsed_s

        Retries on 429 with exponential backoff (5, 10, 20, 40, 80 seconds).
        Raises the last exception if all retries are exhausted.
        """
        for attempt in range(GROK_MAX_RETRIES):
            self._wait_for_rate_limit()
            t0 = time.monotonic()
            try:
                resp = self.client.chat.completions.create(
                    model       = model       or MODEL,
                    max_tokens  = max_tokens  or MAX_TOKENS,
                    temperature = temperature if temperature is not None else TEMPERATURE,
                    messages    = [{'role': 'user', 'content': message}]
                )
                self._last_call_ts = time.monotonic()
                elapsed = self._last_call_ts - t0

                text  = resp.choices[0].message.content
                usage = {
                    'model':         resp.model,
                    'input_tokens':  resp.usage.prompt_tokens,
                    'output_tokens': resp.usage.completion_tokens,
                    'total_tokens':  resp.usage.total_tokens,
                    'elapsed_s':     round(elapsed, 2),
                }
                return text, usage

            except RateLimitError as e:
                wait = 5 * (2 ** attempt)  # 5, 10, 20, 40, 80 seconds
                time.sleep(wait)
                if attempt == GROK_MAX_RETRIES - 1:
                    raise

            except Exception as e:
                raise



## Classify Tickers (Individual Equity, ETF, Mutual Fund, Other)

In [ ]:
def classify_tickers(equity_list, client, model=None):
    """
    Classify tickers as Individual Equity, ETF, Mutual Fund, or Other using a single Grok API call.
    Returns four lists and a geography dict: individual_equities, etfs, mutual_funds, other, geography_map.
    """
    # Create comma-separated list of tickers
    tickers_str = ", ".join(equity_list)
    
    prompt = f"""Classify each of the following tickers. Return TWO values per ticker separated by a pipe (|).

Tickers: {tickers_str}

Column 1 — CLASSIFICATION: Individual Equity, ETF, Mutual Fund, or Other
Column 2 — GEOGRAPHY: one of exactly these values:
  Domestic Stock, Domestic Bond, International Stock, International Bond,
  Commodities, Cash, Pending, N/A

Classification rules:
- US Treasury bonds (9-char CUSIPs like 91282CGS4): Other | Domestic Bond
- Money market funds (**-suffix): Other | Cash
- Cash sweep accounts (**-suffix): Other | Cash
- Options (symbol starts with -): Individual Equity | Domestic Stock
- Pending activity rows: Other | Pending
- GLD, URA: ETF | Commodities
- EMBX: ETF | International Bond
- VEU, VWO, BBJP, FRDM, RNMBY: ETF or Individual Equity | International Stock
- IEF, INCM, AGNC: ETF or Individual Equity | Domestic Bond
- All other US-listed ETFs and stocks: use your judgement for Domestic or International

Return the results in this exact format, one per line:
TICKER: CLASSIFICATION | GEOGRAPHY

Example:
AAPL: Individual Equity | Domestic Stock
SPY: ETF | Domestic Stock
VEU: ETF | International Stock
GLD: ETF | Commodities
91282CGS4: Other | Domestic Bond

Classify each ticker now:"""

    try:
        response_text, _ = client.chat(prompt, model=model or MODEL)
        
        # Parse the response
        individual_equities = []
        etfs = []
        mutual_funds = []
        other = []
        
        # geography_map stores ticker -> geography for use in export cell
        geography_map = {}

        for line in response_text.strip().split('\n'):
            line = line.strip()
            if not line or ':' not in line:
                continue

            # Parse "TICKER: CLASSIFICATION | GEOGRAPHY"
            parts = line.split(':', 1)
            if len(parts) != 2:
                continue

            ticker = parts[0].strip().upper()
            rest   = parts[1].strip()

            # Split on pipe for geography
            if '|' in rest:
                classification, geography = [x.strip() for x in rest.split('|', 1)]
            else:
                classification = rest
                geography      = 'N/A'

            classification = classification.lower()

            # Only process tickers that are in our original list
            if ticker not in [t.upper() for t in equity_list]:
                continue

            # Find the original case ticker
            original_ticker = next((t for t in equity_list if t.upper() == ticker), ticker)

            geography_map[original_ticker] = geography

            if "etf" in classification:
                etfs.append(original_ticker)
            elif "mutual fund" in classification:
                mutual_funds.append(original_ticker)
            elif "individual" in classification or "equity" in classification:
                individual_equities.append(original_ticker)
            else:
                other.append(original_ticker)
        
        # Check for any tickers that weren't classified
        classified = set(t.upper() for t in individual_equities + etfs + mutual_funds + other)
        for ticker in equity_list:
            if ticker.upper() not in classified:
                other.append(ticker)
        
        return individual_equities, etfs, mutual_funds, other, geography_map
        
    except Exception as e:
        # Return all as other if API call fails
        return [], [], [], equity_list, {}

In [ ]:
# ── Classify tickers (with cache) ───────────────────────────────────────
# Grok classification is cached to disk so it's only re-called when the
# ticker list changes. Saves ~20s on every subsequent run with the same portfolio.

_CACHE_FILE = os.path.join(
    os.getenv('Output_dir_portfolio',
              os.path.join(os.path.dirname(PORTFOLIO_CSV_FILE), '..', 'Output_Files', 'Portfolio')),
    'classification_cache.json'
)

def _load_classification_cache():
    """Load cached classification if it exists and ticker list matches."""
    if not os.path.exists(_CACHE_FILE):
        return None
    try:
        with open(_CACHE_FILE) as f:
            cache = json.load(f)
        cached_tickers = sorted(cache.get('tickers', []))
        current_tickers = sorted(EQUITY_LIST)
        if cached_tickers != current_tickers:
            return None
        return cache
    except Exception as e:
        return None

def _save_classification_cache(individual_equities, etfs, mutual_funds, other, geography_map):
    """Save classification result to disk for reuse on future runs."""
    os.makedirs(os.path.dirname(_CACHE_FILE), exist_ok=True)
    cache = {
        'tickers':             sorted(EQUITY_LIST),
        'cached_at':           datetime.now().isoformat(),
        'individual_equities': individual_equities,
        'etfs':                etfs,
        'mutual_funds':        mutual_funds,
        'other':               other,
        'geography_map':       geography_map,
    }
    with open(_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

# Try cache first
client = GrokClient(api_key=API_KEY)
_cache = _load_classification_cache()

if _cache:
    individual_equities = _cache['individual_equities']
    etfs                = _cache['etfs']
    mutual_funds        = _cache['mutual_funds']
    other               = _cache['other']
    geography_map       = _cache['geography_map']
else:
    individual_equities, etfs, mutual_funds, other, geography_map = \
        classify_tickers(EQUITY_LIST, client)
    _save_classification_cache(individual_equities, etfs, mutual_funds, other, geography_map)



## Export Cleaned Portfolio (XLSX + CSV)

In [ ]:
# ============================================================
# Export Cleaned Portfolio Excel File
# ============================================================
# Reads the original Fidelity CSV, removes disclaimer/blank rows,
# cleans numeric formatting, adds Asset Type classification, and
# saves a formatted .xlsx to Output_Files/Portfolio/.
#
# Formatting applied:
#   - Bold, dark-background header row with white text
#   - Alternating light blue / white row shading
#   - Auto-sized column widths
#   - Currency columns right-aligned
#   - Freeze top row

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import re as _re

def _clean_currency(val):
    """
    Convert Fidelity currency/percent strings to float.
    '$1,234.56' -> 1234.56  |  '($780.90)' -> -780.90  |  '--' -> None
    """
    if pd.isna(val): return None
    s = str(val).strip()
    if s in ('', '--', 'NaN'): return None
    negative = s.startswith('(') and s.endswith(')')
    s = s.strip('()').replace('$', '').replace(',', '').replace('%', '').strip()
    try:
        v = float(s)
        return -v if negative else v
    except ValueError:
        return None

def _classify_asset(symbol, description):
    """Classify a portfolio row into a human-readable asset type."""
    sym  = str(symbol).strip()
    desc = str(description).strip().upper()
    if pd.isna(symbol) or sym in ('', 'nan'): return 'Unknown'
    if sym.endswith('**'):
        if 'MONEY MARKET' in desc: return 'Money Market'
        if 'FCASH' in sym or 'HELD IN FCASH' in desc: return 'Cash'
        if 'FDIC' in desc or 'DEPOSIT SWEEP' in desc: return 'Cash'
        return 'Cash'
    if _re.match(r'^[A-Z0-9]{9}$', sym): return 'US Treasury'
    if _re.match(r'^-[A-Z]+\d', sym) or 'CALL' in desc or 'PUT' in desc: return 'Option'
    if 'Pending' in sym: return 'Pending'
    if sym in individual_equities: return 'Individual Equity'
    if sym in etfs: return 'ETF'
    if sym in mutual_funds: return 'Mutual Fund'
    if any(x in desc for x in ['ETF','TRUST','INDEX FDS','ISHARES','VANGUARD',
                                 'SPDR','GLOBAL X','VANECK','SCHWAB','FRANKLIN']):
        return 'ETF'
    return 'Individual Equity'

# ── Load and clean data ───────────────────────────────────────────────────
raw_df = pd.read_csv(PORTFOLIO_CSV_FILE, encoding='utf-8-sig')

# Remove disclaimer/blank rows — keep only rows with a valid numeric Account Number
df = raw_df[raw_df['Account Number'].apply(
    lambda x: str(x).strip().isdigit() if pd.notna(x) else False
)].copy().reset_index(drop=True)

# Rename columns
df = df.rename(columns={
    'Account Number':            'Account Number',
    'Account Name':              'Account Name',
    'Symbol':                    'Symbol',
    'Description':               'Description',
    'Quantity':                  'Quantity',
    'Last Price':                'Last Price ($)',
    'Last Price Change':         'Last Price Change ($)',
    'Current Value':             'Current Value ($)',
    "Today's Gain/Loss Dollar":  "Today Gain/Loss ($)",
    "Today's Gain/Loss Percent": "Today Gain/Loss (%)",
    'Total Gain/Loss Dollar':    'Total Gain/Loss ($)',
    'Total Gain/Loss Percent':   'Total Gain/Loss (%)',
    'Percent Of Account':        'Pct of Account (%)',
    'Cost Basis Total':          'Cost Basis ($)',
    'Average Cost Basis':        'Avg Cost Basis ($)',
    'Type':                      'Type',
})

# Add Asset Type column after Symbol
df.insert(3, 'Asset Type',
    df.apply(lambda r: _classify_asset(r['Symbol'], r['Description']), axis=1)
)

# Add Geography column after Asset Type
# Uses Grok-classified geography_map; falls back to heuristics for non-investable rows
_GEO_HEURISTIC = {
    'Money Market': 'Cash',
    'Cash':         'Cash',
    'Pending':      'Pending',
    'US Treasury':  'Domestic Bond',
    'Option':       'Domestic Stock',
    'Unknown':      'N/A',
}

def _get_geography(symbol, asset_type):
    sym = str(symbol).strip()
    # Heuristic override for non-investable asset types
    if asset_type in _GEO_HEURISTIC:
        return _GEO_HEURISTIC[asset_type]
    # Use Grok-classified result if available
    if sym in geography_map:
        return geography_map[sym]
    return 'N/A'

df.insert(4, 'Allocation 3 Fund',
    df.apply(lambda r: _get_geography(r['Symbol'], r['Asset Type']), axis=1)
)

# Clean numeric columns
currency_cols = ['Last Price ($)','Last Price Change ($)','Current Value ($)',
                 'Today Gain/Loss ($)','Total Gain/Loss ($)','Cost Basis ($)','Avg Cost Basis ($)']
pct_cols      = ['Today Gain/Loss (%)','Total Gain/Loss (%)','Pct of Account (%)']
for col in currency_cols + pct_cols:
    if col in df.columns:
        df[col] = df[col].apply(_clean_currency)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Strip whitespace
for col in ['Account Name','Symbol','Description','Type']:
    df[col] = df[col].astype(str).str.strip()

# ── Build Excel workbook ──────────────────────────────────────────────────
wb = Workbook()
ws = wb.active
ws.title = 'Portfolio'

# Styles
HEADER_FILL  = PatternFill('solid', fgColor='1F3864')   # dark navy
ROW_ALT_FILL = PatternFill('solid', fgColor='DCE6F1')   # light blue
ROW_DEF_FILL = PatternFill('solid', fgColor='FFFFFF')   # white
HEADER_FONT  = Font(bold=True, color='FFFFFF', size=11)
BODY_FONT    = Font(size=10)
CENTER       = Alignment(horizontal='center', vertical='center', wrap_text=False)
RIGHT        = Alignment(horizontal='right',  vertical='center')
LEFT         = Alignment(horizontal='left',   vertical='center')
THIN         = Side(border_style='thin', color='B8CCE4')
BORDER       = Border(bottom=THIN)

# Identify which columns are numeric
num_cols = set(currency_cols + pct_cols + ['Quantity'])

# Write header row
headers = list(df.columns)
for col_idx, header in enumerate(headers, start=1):
    cell             = ws.cell(row=1, column=col_idx, value=header)
    cell.font        = HEADER_FONT
    cell.fill        = HEADER_FILL
    cell.alignment   = CENTER
    cell.border      = BORDER

# Write data rows
for row_idx, row in enumerate(df.itertuples(index=False), start=2):
    fill = ROW_ALT_FILL if row_idx % 2 == 0 else ROW_DEF_FILL
    for col_idx, (header, value) in enumerate(zip(headers, row), start=1):
        cell           = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.font      = BODY_FONT
        cell.fill      = fill
        cell.border    = BORDER
        if header in currency_cols:
            cell.number_format = '#,##0.00'
            cell.alignment     = RIGHT
        elif header in pct_cols:
            cell.number_format = '0.00'
            cell.alignment     = RIGHT
        elif header == 'Quantity':
            cell.number_format = '#,##0.000'
            cell.alignment     = RIGHT
        else:
            cell.alignment     = LEFT

# Auto-size columns
for col_idx, header in enumerate(headers, start=1):
    col_letter  = get_column_letter(col_idx)
    max_width   = len(str(header))
    for row_idx in range(2, ws.max_row + 1):
        val = ws.cell(row=row_idx, column=col_idx).value
        if val is not None:
            max_width = max(max_width, len(str(val)))
    ws.column_dimensions[col_letter].width = min(max_width + 3, 40)

# Freeze header row
ws.freeze_panes = 'A2'

# ── Save ─────────────────────────────────────────────────────────────────
date_str   = datetime.now().strftime('%Y-%m-%d')
output_dir = os.getenv('Output_dir_portfolio',
             os.path.join(os.path.dirname(PORTFOLIO_CSV_FILE), '..', 'Output_Files', 'Portfolio'))
os.makedirs(output_dir, exist_ok=True)
xlsx_path  = os.path.join(output_dir, f'portfolio_cleaned_{date_str}.xlsx')

# ── Totals row ───────────────────────────────────────────────────────────────
TOTAL_COLS = ['Current Value ($)', 'Cost Basis ($)']
total_row_idx = ws.max_row + 1

for col_idx, header in enumerate(headers, start=1):
    cell        = ws.cell(row=total_row_idx, column=col_idx)
    cell.font   = HEADER_FONT
    cell.fill   = HEADER_FILL
    cell.border = BORDER
    if header == 'Symbol':
        cell.value     = 'TOTAL'
        cell.alignment = LEFT
    elif header in TOTAL_COLS:
        cell.value         = round(df[header].sum(skipna=True), 2)
        cell.number_format = '#,##0.00'
        cell.alignment     = RIGHT
    else:
        cell.alignment = LEFT

# ── Allocation Summary pivot sheet ───────────────────────────────────────────
ws2 = wb.create_sheet(title='Allocation Summary')

pivot = (
    df.groupby('Allocation 3 Fund')['Current Value ($)']
    .sum()
    .reset_index()
    .rename(columns={'Allocation 3 Fund': 'Allocation 3 Fund', 'Current Value ($)': 'Current Value ($)'})
    .sort_values('Current Value ($)', ascending=False)
    .reset_index(drop=True)
)

# Add percentage column to pivot
total_value = pivot['Current Value ($)'].sum()
pivot['Current Value (%)'] = (pivot['Current Value ($)'] / total_value * 100).round(0).astype(int)

pivot_headers = ['Allocation 3 Fund', 'Current Value ($)', 'Current Value (%)']

# Header row
for col_idx, header in enumerate(pivot_headers, start=1):
    cell            = ws2.cell(row=1, column=col_idx, value=header)
    cell.font       = HEADER_FONT
    cell.fill       = HEADER_FILL
    cell.alignment  = CENTER
    cell.border     = BORDER

# Data rows
for row_idx, row in enumerate(pivot.itertuples(index=False), start=2):
    fill = ROW_ALT_FILL if row_idx % 2 == 0 else ROW_DEF_FILL
    # Allocation 3 Fund label
    c1            = ws2.cell(row=row_idx, column=1, value=row[0])
    c1.font       = BODY_FONT
    c1.fill       = fill
    c1.border     = BORDER
    c1.alignment  = LEFT
    # Current Value ($)
    c2                = ws2.cell(row=row_idx, column=2, value=round(row[1], 2))
    c2.font           = BODY_FONT
    c2.fill           = fill
    c2.border         = BORDER
    c2.number_format  = '#,##0.00'
    c2.alignment      = RIGHT
    # Current Value (%)
    c3                = ws2.cell(row=row_idx, column=3, value=int(row[2]))
    c3.font           = BODY_FONT
    c3.fill           = fill
    c3.border         = BORDER
    c3.number_format  = '0"%"'
    c3.alignment      = RIGHT

# Total row
total_row = len(pivot) + 2
c1                = ws2.cell(row=total_row, column=1, value='TOTAL')
c1.font           = HEADER_FONT
c1.fill           = HEADER_FILL
c1.border         = BORDER
c1.alignment      = LEFT
c2                = ws2.cell(row=total_row, column=2, value=round(pivot['Current Value ($)'].sum(), 2))
c2.font           = HEADER_FONT
c2.fill           = HEADER_FILL
c2.border         = BORDER
c2.number_format  = '#,##0.00'
c2.alignment      = RIGHT
c3                = ws2.cell(row=total_row, column=3, value=100)
c3.font           = HEADER_FONT
c3.fill           = HEADER_FILL
c3.border         = BORDER
c3.number_format  = '0"%"'
c3.alignment      = RIGHT

# Auto-size columns
for col_idx, header in enumerate(pivot_headers, start=1):
    col_letter = get_column_letter(col_idx)
    max_width  = len(header)
    for row_idx in range(2, ws2.max_row + 1):
        val = ws2.cell(row=row_idx, column=col_idx).value
        if val is not None:
            max_width = max(max_width, len(str(val)))
    ws2.column_dimensions[col_letter].width = min(max_width + 3, 40)

ws2.freeze_panes = 'A2'

wb.save(xlsx_path)

# Also save a plain CSV alongside for downstream scripts / other agents
csv_path = xlsx_path.replace('.xlsx', '.csv')
df.to_csv(csv_path, index=False)



## Shared SEC Helper Functions

In [ ]:
# ============================================================
# Shared SEC Helper Functions (Priorities 1 & 2)
# ============================================================
# Previously, the notebook downloaded the SEC ticker JSON files
# once per ticker inside each processing loop — causing the same
# ~2 MB files to be fetched dozens of times.  These helpers:
#   1. Download each SEC JSON file exactly once (load_sec_data_once)
#   2. Provide a single CIK lookup used by all three asset types
#   3. Provide a single filing-fetch function used by all three loops
#   4. Provide a single filing-download function used by all three loops

SEC_REQUEST_DELAY = 0.15   # seconds between requests (~6.6 req/s, under EDGAR 10 req/s limit)
SEC_MAX_RETRIES   = 3

# SEC form sets per asset type
EQUITY_FORMS = ['10-K', '10-Q', '8-K']

FUND_FORMS = {
    'N-1A':    'N-1A_Prospectus_SAI',
    'N-1A/A':  'N-1A_Amendment',
    '485BPOS': 'N-1A_Post_Effective_Amend',
    '485APOS': 'N-1A_Pre_Effective_Amend',
    'N-CSR':   'Annual_Shareholder_Report',
    'N-CSRS':  'Semiannual_Shareholder_Report',
    'N-PORT':  'Portfolio_Holdings',
    'N-PORT-P':'Portfolio_Holdings',
    'N-CEN':   'Annual_Report_N-CEN',
}

def sec_request_with_retry(url, headers,
                            max_retries=SEC_MAX_RETRIES,
                            delay=SEC_REQUEST_DELAY):
    """GET with rate-limiting delay and exponential-backoff retry."""
    for attempt in range(max_retries):
        try:
            time.sleep(delay)
            resp = requests.get(url, headers=headers, timeout=30)
            return resp
        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                ConnectionResetError) as e:
            wait = (attempt + 1) * 2
            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                return None
    return None

# Local cache paths for SEC JSON lookup files (invalidated after 24h)
_SEC_CACHE_DIR      = None   # set at runtime from OUTPUT_DIR_SEC_FILINGS
_SEC_CACHE_MAX_AGE  = 86400  # 24 hours in seconds

def _sec_cache_path(filename):
    base = _SEC_CACHE_DIR or OUTPUT_DIR_SEC_FILINGS or '/tmp'
    return os.path.join(base, filename)

def _load_sec_json_cached(url, cache_filename, headers):
    """
    Fetch a SEC JSON file, caching it locally for up to 24 hours.
    Avoids re-downloading large files (~2 MB each) on every run.
    """
    cache_path = _sec_cache_path(cache_filename)
    # Use cache if it exists and is fresh
    if os.path.exists(cache_path):
        age = time.time() - os.path.getmtime(cache_path)
        if age < _SEC_CACHE_MAX_AGE:
            with open(cache_path) as f:
                return json.load(f)
        else:
            pass  # cache expired — fall through to re-download
    # Download fresh
    resp = sec_request_with_retry(url, headers)
    if resp and resp.status_code == 200:
        data = resp.json()
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, 'w') as f:
            json.dump(data, f)
        return data
    return None

def load_sec_data_once(headers):
    """
    Load SEC ticker lookup files, using a 24h local disk cache.
    Returns (exchange_data, mf_data) — either may be None on failure.
    """
    exchange_data = _load_sec_json_cached(
        'https://www.sec.gov/files/company_tickers_exchange.json',
        'sec_exchange_tickers.json', headers)
    mf_data = _load_sec_json_cached(
        'https://www.sec.gov/files/company_tickers_mf.json',
        'sec_mf_tickers.json', headers)
    eq_count = len(exchange_data.get('data', [])) if exchange_data else 0
    mf_count = len(mf_data.get('data', []))      if mf_data      else 0
    return exchange_data, mf_data

def build_cik_index(exchange_data, mf_data=None):
    """
    Pre-build a dict {ticker_upper -> zero_padded_cik} from SEC data for O(1) lookups.
    Call once after load_sec_data_once() and reuse across all tickers.
    """
    index = {}
    for data, sym_field, cik_field_substr in [
        (exchange_data, 'ticker', 'cik'),
        (mf_data,       'symbol', 'cik'),
    ]:
        if not data:
            continue
        fields    = data.get('fields', [])
        data_list = data.get('data',   [])
        try:
            sym_idx = fields.index(sym_field)
            cik_idx = next(i for i, f in enumerate(fields) if cik_field_substr in f.lower())
        except (ValueError, StopIteration):
            continue
        for row in data_list:
            sym = str(row[sym_idx]).upper()
            if sym not in index:
                index[sym] = str(row[cik_idx]).zfill(10)
    return index

def fetch_cik_from_data(ticker, exchange_data, mf_data=None, cik_index=None):
    """
    Look up CIK for a ticker. Uses pre-built index for O(1) lookup if available,
    otherwise falls back to linear scan.
    Returns zero-padded CIK string or None.
    """
    if cik_index is not None:
        return cik_index.get(ticker.upper())

    def _search(data, symbol_field, cik_field):
        if not data:
            return None
        fields    = data.get('fields', [])
        data_list = data.get('data',   [])
        try:
            sym_idx = fields.index(symbol_field)
            cik_idx = next(i for i, f in enumerate(fields) if cik_field in f.lower())
        except (ValueError, StopIteration):
            return None
        for row in data_list:
            if str(row[sym_idx]).upper() == ticker.upper():
                return str(row[cik_idx]).zfill(10)
        return None

    cik = _search(exchange_data, 'ticker', 'cik')
    if not cik and mf_data:
        cik = _search(mf_data, 'symbol', 'cik')
    return cik

def fetch_filings_for_cik(cik, target_forms, headers):
    """
    Fetch the most recent filing of each requested form type for a given CIK.
    target_forms: list (equity) or dict (funds, where keys are form codes).
    Returns a list of filing dicts with keys: form, date, url, [type].
    """
    url  = f'https://data.sec.gov/submissions/CIK{cik}.json'
    resp = sec_request_with_retry(url, headers)
    if not resp or resp.status_code != 200:
        return []

    data         = resp.json()
    recent       = data.get('filings', {}).get('recent', {})
    forms_list   = recent.get('form',            [])
    accessions   = recent.get('accessionNumber', [])
    dates        = recent.get('filingDate',      [])
    primary_docs = recent.get('primaryDocument', [])

    is_fund      = isinstance(target_forms, dict)
    form_set     = set(target_forms.keys()) if is_fund else set(target_forms)
    found        = set()
    filings      = []

    for i, form in enumerate(forms_list):
        if form in form_set and form not in found:
            acc_clean = accessions[i].replace('-', '')
            filing = {
                'form': form,
                'date': dates[i],
                'url':  f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_clean}/{primary_docs[i]}',
            }
            if is_fund:
                filing['type'] = target_forms[form]
            filings.append(filing)
            found.add(form)
            if len(found) == len(form_set):
                break

    return filings

def download_filings_for_ticker(ticker, filings, output_dir, headers):
    """
    Download a list of SEC filings for a ticker to output_dir.
    Skips tickers that already have downloaded files (avoids re-downloading on repeat runs).
    Works for equities (no 'type' key) and funds (has 'type' key).
    Returns the number of files saved (0 if all skipped).
    """
    os.makedirs(output_dir, exist_ok=True)
    # #1: Skip download entirely if this ticker already has files in output_dir
    existing = [f for f in os.listdir(output_dir)
                if f.startswith(f'{ticker}_') and f.endswith('.html')]
    if existing:
        return 0
    saved = 0
    for filing in filings:
        form     = filing['form'].replace('/', '-')
        date     = filing['date']
        doc_type = filing.get('type', form).replace(' ', '_')
        filename = f'{ticker}_{doc_type}_{form}_{date}.html'
        filepath = os.path.join(output_dir, filename)

        resp = sec_request_with_retry(filing['url'], headers)
        if resp and resp.status_code == 200:
            with open(filepath, 'wb') as fh:
                fh.write(resp.content)
            saved += 1
        else:
            pass
    return saved

def process_tickers_sec(ticker_list, asset_type, cik_cache,
                         exchange_data, mf_data, target_forms,
                         output_dir, headers, cik_index=None):
    """
    Unified SEC processing loop for any asset type.
    Replaces the three near-identical loops that previously existed for
    individual equities, ETFs, and mutual funds.

    Parameters
    ----------
    ticker_list   : list of tickers to process
    asset_type    : label string for logging ('Individual Equity', 'ETF', 'Mutual Fund')
    cik_cache     : dict {ticker -> cik} loaded from cik_lookup.json; updated in-place
    exchange_data : pre-fetched company_tickers_exchange.json dict
    mf_data       : pre-fetched company_tickers_mf.json dict
    target_forms  : list (equities) or dict (funds) of SEC forms to fetch
    output_dir    : directory to save downloaded filings
    headers       : HTTP headers dict with User-Agent for SEC EDGAR

    Returns
    -------
    Updated cik_cache dict
    """

    fetched_count = 0
    cached_count  = 0

    for ticker in ticker_list:

        # CIK lookup: cache first, then SEC data
        if ticker in cik_cache:
            cik = cik_cache[ticker]
            cached_count += 1
        else:
            cik = fetch_cik_from_data(ticker, exchange_data, mf_data, cik_index=cik_index)
            if cik:
                cik_cache[ticker] = cik
                fetched_count += 1
            else:
                continue

        # Fetch and download filings
        filings = fetch_filings_for_cik(cik, target_forms, headers)
        download_filings_for_ticker(ticker, filings, output_dir, headers)

    return cik_cache



## Run SEC CIK Lookup and Filing Downloads

In [ ]:
# ============================================================
# Run SEC CIK Lookup and Filing Downloads (All Asset Types)
# ============================================================
# A single call to process_tickers_sec() replaces the three
# near-identical processing loops that previously existed.
# SEC ticker JSON files are downloaded exactly once via
# load_sec_data_once(), not once per ticker.

headers = {'User-Agent': SEC_HEADER}

# Point SEC JSON cache to the filings directory (shared with CIK cache)
_SEC_CACHE_DIR = OUTPUT_DIR_SEC_FILINGS
os.makedirs(OUTPUT_DIR_SEC_FILINGS, exist_ok=True)

# Load the SEC ticker lookup files once (Priority 2 fix)
exchange_data, mf_data = load_sec_data_once(headers)

# Load existing CIK cache
cik_lookup_file = os.path.join(OUTPUT_DIR_SEC_FILINGS, 'cik_lookup.json')
existing_cik_lookup = {'individual_equities': {}, 'etfs': {}, 'mutual_funds': {}}
if os.path.exists(cik_lookup_file):
    with open(cik_lookup_file, 'r') as f:
        existing_cik_lookup = json.load(f)
else:
    pass
os.makedirs(OUTPUT_DIR_SEC_FILINGS, exist_ok=True)

# Process all three asset types using the shared function (Priority 1 fix)
equity_ciks = process_tickers_sec(
    ticker_list   = individual_equities,
    asset_type    = 'Individual Equity',
    cik_cache     = dict(existing_cik_lookup.get('individual_equities', {})),
    exchange_data = exchange_data,
    mf_data       = mf_data,
    target_forms  = EQUITY_FORMS,
    output_dir    = OUTPUT_DIR_SEC_FILINGS,
    headers       = headers,
)

etf_ciks = process_tickers_sec(
    ticker_list   = etfs,
    asset_type    = 'ETF',
    cik_cache     = dict(existing_cik_lookup.get('etfs', {})),
    exchange_data = exchange_data,
    mf_data       = mf_data,
    target_forms  = FUND_FORMS,
    output_dir    = OUTPUT_DIR_SEC_FILINGS,
    headers       = headers,
)

mutual_fund_ciks = process_tickers_sec(
    ticker_list   = mutual_funds,
    asset_type    = 'Mutual Fund',
    cik_cache     = dict(existing_cik_lookup.get('mutual_funds', {})),
    exchange_data = exchange_data,
    mf_data       = mf_data,
    target_forms  = FUND_FORMS,
    output_dir    = OUTPUT_DIR_SEC_FILINGS,
    headers       = headers,
)

# Save updated CIK cache
cik_lookup = {
    'individual_equities': equity_ciks,
    'etfs':                etf_ciks,
    'mutual_funds':        mutual_fund_ciks,
    'metadata': {
        'generated_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_equities':     len(equity_ciks),
        'total_etfs':         len(etf_ciks),
        'total_mutual_funds': len(mutual_fund_ciks),
    }
}
with open(cik_lookup_file, 'w') as f:
    json.dump(cik_lookup, f, indent=2)

# Also save a flat CSV for easy reference
cik_rows = (
    [{'ticker': t, 'cik': c, 'type': 'Individual Equity'} for t, c in equity_ciks.items()] +
    [{'ticker': t, 'cik': c, 'type': 'ETF'}               for t, c in etf_ciks.items()] +
    [{'ticker': t, 'cik': c, 'type': 'Mutual Fund'}        for t, c in mutual_fund_ciks.items()]
)
cik_csv = os.path.join(OUTPUT_DIR_SEC_FILINGS, 'cik_lookup.csv')
pd.DataFrame(cik_rows).to_csv(cik_csv, index=False)


## Word Document Formatting Functions

In [ ]:
# ============================================================
# Word Document Formatting — Refactored (Priority #6)
# ============================================================
# Previously add_markdown_to_word() was ~200 lines of tangled logic
# with nested helpers, lookahead, and HTML parsing all mixed together.
# Now split into single-responsibility functions:
#   _clean_html()         — strip HTML tags from any string
#   _add_formatted_run()  — add bold/italic/code runs to a paragraph
#   _handle_heading()     — process # ## ### lines
#   _handle_table()       — process | table | lines
#   _handle_bullet()      — process - / * / bullet lines
#   _handle_numbered()    — process 1. 2. numbered list lines
#   _handle_styled_bold() — process **<span>...</span>** styled headers
#   _handle_paragraph()   — process all remaining plain text
#   add_markdown_to_word()— thin dispatcher loop calling the above

# ── Shared utilities ──────────────────────────────────────────────────────

def _clean_html(text: str) -> str:
    """Strip all HTML tags from text."""
    return re.sub(r'<[^>]+>', '', text)

def _extract_styled_span(line: str):
    """
    Extract inner text from **<span style='...'>text</span>** patterns.
    Returns (text, True) if found, (None, False) otherwise.
    """
    for pattern in (
        r'\*\*<span[^>]*>(.+?)</span>\*\*',  # **<span>text</span>**
        r'<span[^>]*>(.+?)</span>',            # bare <span>text</span>
    ):
        m = re.search(pattern, line)
        if m:
            return m.group(1).strip(), True
    return None, False

def _add_formatted_run(paragraph, text: str, allow_bold: bool = False) -> None:
    """
    Add text to a paragraph, honouring **bold**, __bold__, and `code` markers.
    HTML tags are stripped before processing.
    """
    text = _clean_html(text)
    for part in re.split(r'(\*\*.*?\*\*|__.*?__|`.*?`)', text):
        if not part:
            continue
        if (part.startswith('**') and part.endswith('**')) or \
           (part.startswith('__') and part.endswith('__')):
            run = paragraph.add_run(part[2:-2])
            run.bold = allow_bold
        elif part.startswith('`') and part.endswith('`'):
            run = paragraph.add_run(part[1:-1])
            run.font.name = 'Courier New'
        else:
            paragraph.add_run(part)

# ── Single-responsibility handlers ────────────────────────────────────────────

def _handle_heading(doc, line: str) -> None:
    """Process a markdown heading line (# ## ###)."""
    level = len(line) - len(line.lstrip('#'))
    text  = _clean_html(line.lstrip('#').strip())
    doc.add_heading(text, level=min(level, 9))

def _handle_styled_bold(doc, line: str) -> bool:
    """
    Process a **<span>...</span>** styled header or plain **bold header** line.
    Returns True if handled, False if the line should fall through.
    """
    styled_text, is_styled = _extract_styled_span(line)
    if is_styled:
        p   = doc.add_paragraph()
        run = p.add_run(styled_text)
        run.bold             = True
        run.font.size        = Pt(14)
        run.font.color.rgb   = RGBColor(0, 0, 139)
        return True

    stripped = line.strip()
    if stripped.startswith('**') and stripped.endswith('**') and len(stripped) < 100:
        p = doc.add_paragraph()
        _add_formatted_run(p, stripped, allow_bold=True)
        return True

    return False

def _handle_table(doc, lines: list, start_idx: int) -> int:
    """
    Process a markdown table starting at start_idx.
    Returns the index of the first line after the table.
    """
    i = start_idx
    table_lines = [lines[i]]
    i += 1
    # Skip separator row (---, :-:, etc.)
    if i < len(lines) and re.match(r'^[|\s\-:]+$', lines[i]):
        i += 1
    # Collect remaining table rows
    while i < len(lines) and '|' in lines[i]:
        table_lines.append(lines[i])
        i += 1

    headers  = [c.strip() for c in table_lines[0].split('|') if c.strip()]
    num_cols = len(headers)
    num_rows = len(table_lines)

    table       = doc.add_table(rows=num_rows, cols=num_cols)
    table.style = 'Light Grid Accent 1'

    # Header row
    for j, header in enumerate(headers):
        cell      = table.rows[0].cells[j]
        cell.text = _clean_html(header.replace('**', '').replace('__', ''))
        if cell.paragraphs[0].runs:
            cell.paragraphs[0].runs[0].bold = True

    # Data rows
    for row_idx in range(1, len(table_lines)):
        cells = [c.strip() for c in table_lines[row_idx].split('|') if c.strip()]
        for col_idx, cell_text in enumerate(cells):
            if col_idx < num_cols:
                table.rows[row_idx].cells[col_idx].text = \
                    _clean_html(cell_text.replace('**', '').replace('__', ''))

    doc.add_paragraph()
    return i

def _handle_bullet(doc, line: str, upcoming_bullets: int) -> None:
    """
    Process a bullet point line (-, *, bullet).
    Uses List Bullet style only when 3+ consecutive bullets exist;
    otherwise renders as a plain paragraph to avoid over-bulleting.
    """
    text = _clean_html(line.strip()[2:].strip())
    if upcoming_bullets >= 3:
        p = doc.add_paragraph(style='List Bullet')
        _add_formatted_run(p, text, allow_bold=False)
    else:
        p = doc.add_paragraph()
        _add_formatted_run(p, text, allow_bold=False)

def _handle_numbered(doc, line: str) -> bool:
    """
    Process a numbered list line (1. 2. etc.).
    Checks for HTML span styling first; falls back to List Number style.
    Returns True if handled.
    """
    if '<span' in line:
        styled_text, is_styled = _extract_styled_span(line)
        if is_styled:
            p   = doc.add_paragraph()
            run = p.add_run(styled_text)
            run.bold           = True
            run.font.size      = Pt(14)
            run.font.color.rgb = RGBColor(0, 0, 139)
            return True

    text = _clean_html(re.sub(r'^\d+\.\s', '', line.strip()))
    p    = doc.add_paragraph(style='List Number')
    _add_formatted_run(p, text, allow_bold=True)
    for run in p.runs:
        run.font.size = Pt(14)
        run.bold      = True
    return True

def _handle_paragraph(doc, line: str) -> None:
    """Process a plain text paragraph line."""
    p = doc.add_paragraph()
    _add_formatted_run(p, _clean_html(line), allow_bold=False)

def _count_upcoming_bullets(lines: list, start_idx: int) -> int:
    """Count consecutive bullet lines from start_idx (skipping blanks)."""
    count = 0
    for j in range(start_idx, len(lines)):
        stripped = lines[j].strip()
        if stripped.startswith(('- ', '* ', '\u2022 ')):
            count += 1
        elif stripped == '':
            continue
        else:
            break
    return count

# ── Main dispatcher ───────────────────────────────────────────────────────────

def add_markdown_to_word(doc, markdown_text: str) -> None:
    """
    Convert markdown text to formatted Word document content.

    Thin dispatcher: classifies each line and delegates to the appropriate
    single-responsibility handler.  Maintains no internal state beyond
    the line index and upcoming-bullet count.
    """
    lines = markdown_text.split('\n')
    i     = 0

    while i < len(lines):
        line    = lines[i]
        stripped = line.strip()

        # Blank line — skip
        if not stripped:
            i += 1
            continue

        # Heading
        if stripped.startswith('#'):
            _handle_heading(doc, stripped)
            i += 1
            continue

        # Table
        if '|' in line and i + 1 < len(lines) and '|' in lines[i + 1]:
            i = _handle_table(doc, lines, i)
            continue

        # Styled bold / span header
        if '<span' in line or (stripped.startswith('**') and stripped.endswith('**') and len(stripped) < 100):
            if _handle_styled_bold(doc, line):
                i += 1
                continue

        # Bullet point
        if stripped.startswith(('- ', '* ', '\u2022 ')):
            upcoming = _count_upcoming_bullets(lines, i)
            _handle_bullet(doc, line, upcoming)
            i += 1
            continue

        # Numbered list
        if re.match(r'^\d+\.\s', stripped):
            _handle_numbered(doc, line)
            i += 1
            continue

        # Plain paragraph (catch-all)
        _handle_paragraph(doc, line)
        i += 1

# Keep add_formatted_text as a public alias for any external callers
add_formatted_text = _add_formatted_run

# ── Table sort utility (unchanged) ────────────────────────────────────────────

def sort_markdown_table_by_industry(markdown_text: str) -> str:
    """Sort the first markdown table in the text by Industry or Category column."""
    lines       = markdown_text.split('\n')
    table_start = next((i for i, l in enumerate(lines) if '|' in l), None)
    if table_start is None:
        return markdown_text
    table_end = next(
        (i for i in range(table_start + 1, len(lines)) if '|' not in lines[i] or not lines[i].strip()),
        len(lines)
    )

    header    = lines[table_start]
    separator = lines[table_start + 1]
    rows      = lines[table_start + 2:table_end]
    columns   = [c.strip().lower() for c in header.split('|')]

    try:
        sort_idx = next(i for i, c in enumerate(columns) if c in ('industry', 'category'))
    except StopIteration:
        return markdown_text

    rows_sorted = sorted(rows, key=lambda r: ([c.strip() for c in r.split('|')] + [''])[sort_idx])
    sorted_lines = lines[:table_start] + [header, separator] + rows_sorted + lines[table_end:]
    return '\n'.join(sorted_lines)



## SEC Filing Index

In [ ]:
# ============================================================
# Build SEC Filing Index (file list + ticker dictionary)
# ============================================================
# Scans the SEC filings output directory, groups files by ticker.
# Skips rebuilding the index if no new files were downloaded this run.

sec_files = []
sec_files_by_ticker = {}
EXCLUDE_PREFIXES = {'cik', 'sec_files', 'sec_filings', 'sec_exchange', 'sec_mf'}

if os.path.exists(OUTPUT_DIR_SEC_FILINGS):
    current_files = sorted(f for f in os.listdir(OUTPUT_DIR_SEC_FILINGS)
                           if not f.startswith('.'))

    # #7: Check if index is already up to date
    dict_path  = os.path.join(OUTPUT_DIR_SEC_FILINGS, 'sec_files_by_ticker.json')
    list_path  = os.path.join(OUTPUT_DIR_SEC_FILINGS, 'sec_filings_list.txt')
    needs_rebuild = True

    if os.path.exists(dict_path) and os.path.exists(list_path):
        with open(list_path) as fh:
            cached_count_line = [l for l in fh.readlines() if l.startswith('Total files:')]
        if cached_count_line:
            cached_count = int(cached_count_line[0].split(':')[1].strip())
            html_files   = [f for f in current_files if f.endswith('.html')]
            if cached_count == len(html_files):
                with open(dict_path) as fh:
                    sec_files_by_ticker = json.load(fh)
                sec_files    = current_files
                needs_rebuild = False

    if needs_rebuild:
        sec_files = current_files
        for fname in sec_files:
            ticker = fname.split('_')[0] if '_' in fname else fname
            prefix = ticker.lower().rstrip('_')
            if any(prefix.startswith(p) for p in EXCLUDE_PREFIXES):
                continue
            sec_files_by_ticker.setdefault(ticker, []).append(fname)

        for t, files in sorted(sec_files_by_ticker.items()):

            pass
        with open(list_path, 'w') as fh:
            fh.write(f'SEC Filings List - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
            fh.write(f'Directory: {OUTPUT_DIR_SEC_FILINGS}\n')
            html_count = len([f for f in sec_files if f.endswith('.html')])
            fh.write(f'Total files: {html_count}\n' + '-'*60 + '\n')
            fh.writelines(f'{f}\n' for f in sec_files if f.endswith('.html'))

        with open(dict_path, 'w') as fh:
            json.dump(sec_files_by_ticker, fh, indent=2)
else:

    pass

## Shared Report Generation Function

In [ ]:
# ============================================================
# Shared Report Generation Function (Priority #9)
# ============================================================
# Previously three near-identical loops existed for Individual
# Equities, ETFs, and Mutual Funds. All shared logic extracted
# into generate_report(), called three times below.

# client already instantiated in the Classify Tickers cell.
# Output dirs are created by generate_report() as needed.

def read_sec_filing_content(filepath: str, max_chars: int = 50000) -> str | None:
    """Read an SEC filing HTML file and return stripped plain text, truncated to max_chars."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        text = re.sub(r'<[^>]+>', ' ', content)
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars] + '... [truncated]' if len(text) > max_chars else text
    except Exception as e:
        return None

def _gather_sec_content(ticker: str, sec_files_by_ticker: dict,
                         sec_filings_dir: str, max_chars: int = 30000):
    """
    Gather and concatenate SEC filing text for a ticker.
    Returns (sec_content_str, list_of_filenames_used).
    """
    # #3: Cache Grok report output keyed on ticker + most recent SEC filing date
    # Re-uses prior output when filings haven't changed between runs
    _cache_dir  = output_dir
    _cache_key  = ticker
    _cache_file = os.path.join(_cache_dir, f'{ticker}_report_cache.json')
    if os.path.exists(_cache_file):
        try:
            with open(_cache_file) as _cf:
                _cached = json.load(_cf)
            # Find the most recent SEC filing mtime for this ticker
            _filing_files = [
                os.path.join(sec_filings_dir, f)
                for f in sec_files_by_ticker.get(ticker, [])
                if os.path.exists(os.path.join(sec_filings_dir, f))
            ]
            _latest_mtime = max((os.path.getmtime(f) for f in _filing_files), default=0)
            if _cached.get('sec_mtime', 0) >= _latest_mtime:
                # Filings unchanged — write docx from cache and return
                _doc = create_word_document(_cached['content'], ticker, asset_type)
                _date_str = datetime.now().strftime('%Y-%m-%d')
                _fname    = f'{asset_type} Report - {ticker} {_date_str}.docx'
                _doc.save(os.path.join(output_dir, _fname))
                return counter
        except Exception:
            pass  # cache miss or corrupt — fall through to regenerate

    sec_content   = ''
    sec_files_used = []
    if ticker in sec_files_by_ticker:

        for sec_file in sec_files_by_ticker[ticker]:
            filepath = os.path.join(sec_filings_dir, sec_file)
            text     = read_sec_filing_content(filepath, max_chars=max_chars)
            if text:
                sec_files_used.append(sec_file)
                sec_content += f'\n\n--- SEC Filing: {sec_file} ---\n{text}'
    return sec_content, sec_files_used

def generate_report(
    ticker:              str,
    asset_type:          str,
    prompt_template:     str,
    output_dir:          str,
    client:              'GrokClient',
    sec_files_by_ticker: dict,
    sec_filings_dir:     str,
    counter:             int  = 0,
    total:               int  = 0,
    include_prompt:      bool = True,
    max_chars:           int  = 50000,
) -> int:
    """
    Generate an analysis Word document for a single ticker.
    Shared by Individual Equity, ETF, and Mutual Fund processing.

    Parameters
    ----------
    ticker              : stock/ETF/fund ticker symbol
    asset_type          : 'equity', 'ETF', or 'Mutual Fund' (used in titles)
    prompt_template     : analysis prompt template string
    output_dir          : directory to save the Word doc
    client              : GrokClient instance
    sec_files_by_ticker : dict mapping ticker -> list of SEC filenames
    sec_filings_dir     : directory containing downloaded SEC filings
    counter             : current item count (for progress logging)
    total               : total items in this batch
    include_prompt      : whether to append the prompt template to the doc

    Returns
    -------
    Updated counter (incremented on success, unchanged on skip/error).
    """
    date_str        = datetime.now().strftime('%Y-%m-%d')
    type_label      = asset_type.replace('_', ' ').title()
    output_filename = f'{type_label} Report - {ticker} {date_str}.docx'
    output_path     = os.path.join(output_dir, output_filename)

    # Skip if report already exists for today
    if os.path.exists(output_path):
        return counter

    # Gather SEC filing content
    sec_content, sec_files_used = _gather_sec_content(
        ticker, sec_files_by_ticker, sec_filings_dir
    )

    # Build prompt
    if sec_content:
        prompt = (
            f'For the {type_label} {ticker}, analyze the following SEC filings '
            f'and {prompt_template}\n\nSEC FILINGS:\n{sec_content}'
        )
    else:
        prompt = f'For the {type_label} {ticker} {prompt_template}'

    try:
        if sec_files_used:
            _sec_joined = ', '.join(sec_files_used)

        generated_text, llm_usage = client.chat(prompt, model=MODEL)
        counter += 1

        # Build Word document
        doc = Document()
        doc.add_heading(f'{type_label} Analysis Report - {ticker}', 0)
        doc.add_paragraph(f'Grok Model Generated: {date_str}')
        doc.add_paragraph()

        if sec_files_used:
            doc.add_heading('SEC Filings Analyzed:', level=2)
            for sf in sec_files_used:
                doc.add_paragraph(sf, style='List Bullet')
            doc.add_paragraph()

        add_markdown_to_word(doc, generated_text)

        if include_prompt:
            doc.add_paragraph()
            doc.add_heading('Prompt Template Used:', level=2)
            summary = f'For the {type_label} {ticker} {prompt_template}'
            if sec_files_used:
                _sec_joined2 = ', '.join(sec_files_used)
            summary += f'\n\n[SEC filings included: {_sec_joined2}]'
            doc.add_paragraph(summary)

        os.makedirs(output_dir, exist_ok=True)
        doc.save(output_path)

    except Exception as e:
        pass

    return counter



## Generate Analysis Reports — Individual Equities

In [ ]:
# Individual Equity reports
# ── Shared import (used by Equity, ETF, and Mutual Fund cells) ──────────────
from concurrent.futures import ThreadPoolExecutor as _TPE, as_completed as _asc
import functools as _ft

# ── Tuesday-only gate ───────────────────────────────────────────────────────
# Individual equity reports are expensive and only regenerated once a week.
# Skip this section entirely unless today is Tuesday (weekday == 1),
# or unless FORCE_EQUITY_REPORTS=1 is set in the environment.
_is_tuesday = datetime.now().weekday() == 1  # 0=Mon, 1=Tue, ...
_force      = os.getenv('FORCE_EQUITY_REPORTS', '0') == '1'

if not (_is_tuesday or _force):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Skipping Individual Equity reports "
          f"(today is {datetime.now().strftime('%A')} — only runs on Tuesdays). "
          f"Set FORCE_EQUITY_REPORTS=1 to override.")
else:
    counter = 0

    # Upfront check — skip entire loop if all today's reports already exist
    _date_str = datetime.now().strftime('%Y-%m-%d')
    _expected = [f'Equity Report - {t} {_date_str}.docx' for t in individual_equities]
    _missing  = [f for f in _expected if not os.path.exists(os.path.join(OUTPUT_DIR_INDIVIDUAL_STOCK_ANALYSIS, f))]
    if not _missing:
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] All Individual Equity reports already exist for today — skipping.")
    else:
        def _gen_equity(equity):
            return generate_report(
                ticker              = equity,
                asset_type          = 'Equity',
                prompt_template     = PROMPT_INDIVIDUAL_EQUITY_TEMPLATE,
                output_dir          = OUTPUT_DIR_INDIVIDUAL_STOCK_ANALYSIS,
                client              = client,
                sec_files_by_ticker = sec_files_by_ticker,
                sec_filings_dir     = OUTPUT_DIR_SEC_FILINGS
            )

        with _TPE(max_workers=4) as _pool:
            list(_pool.map(_gen_equity, individual_equities))


## Generate Analysis Reports — ETFs

In [ ]:
# ETF reports
OUTPUT_DIR_ETF_ANALYSIS = os.getenv(
    'Output_dir_etf_analysis', OUTPUT_DIR_INDIVIDUAL_STOCK_ANALYSIS
)
os.makedirs(OUTPUT_DIR_ETF_ANALYSIS, exist_ok=True)

# ── Tuesday-only gate ───────────────────────────────────────────────────────
# ETF reports are only regenerated once a week on Tuesdays.
# Skip unless today is Tuesday or FORCE_ETF_REPORTS=1 is set.
_is_tuesday = datetime.now().weekday() == 1
_force_etf  = os.getenv('FORCE_ETF_REPORTS', '0') == '1'

if not (_is_tuesday or _force_etf):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Skipping ETF reports "
          f"(today is {datetime.now().strftime('%A')} — only runs on Tuesdays). "
          f"Set FORCE_ETF_REPORTS=1 to override.")
else:
    etf_counter = 0

    # Upfront check — skip entire loop if all today's reports already exist
    _date_str = datetime.now().strftime('%Y-%m-%d')
    _expected = [f'Etf Report - {t} {_date_str}.docx' for t in etfs]
    _missing  = [f for f in _expected if not os.path.exists(os.path.join(OUTPUT_DIR_ETF_ANALYSIS, f))]
    if not _missing:
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] All ETF reports already exist for today — skipping.")
    else:
        # Parallel ETF report generation
        def _gen_etf(etf):
            return generate_report(
                ticker              = etf,
                asset_type          = 'ETF',
                prompt_template     = PROMPT_ETF_ANALYSIS_FILE_TEMPLATE,
                output_dir          = OUTPUT_DIR_ETF_ANALYSIS,
                client              = client,
                sec_files_by_ticker = sec_files_by_ticker,
                sec_filings_dir     = OUTPUT_DIR_SEC_FILINGS
            )

        with _TPE(max_workers=4) as _pool:
            list(_pool.map(_gen_etf, etfs))


## Generate Analysis Reports — Mutual Funds

In [ ]:
# Mutual Fund reports
OUTPUT_DIR_MUTUAL_FUND_ANALYSIS = os.getenv(
    'Output_dir_mutual_fund_analysis', OUTPUT_DIR_INDIVIDUAL_STOCK_ANALYSIS
)
os.makedirs(OUTPUT_DIR_MUTUAL_FUND_ANALYSIS, exist_ok=True)
# Mutual funds share the ETF analysis prompt template.
mf_counter = 0

# #3: Upfront check — skip entire loop if all today's reports already exist
_date_str = datetime.now().strftime('%Y-%m-%d')
_expected = [f'Mutual Fund Report - {t} {_date_str}.docx' for t in mutual_funds]
_missing  = [f for f in _expected if not os.path.exists(os.path.join(OUTPUT_DIR_MUTUAL_FUND_ANALYSIS, f))]
if not _missing:
    pass
else:

    pass
# #1: Parallel Mutual Fund report generation
def _gen_mf(mf):
    return generate_report(
        ticker              = mf,
        asset_type          = 'Mutual Fund',
        prompt_template     = PROMPT_ETF_ANALYSIS_FILE_TEMPLATE,
        output_dir          = OUTPUT_DIR_MUTUAL_FUND_ANALYSIS,
        client              = client,
        sec_files_by_ticker = sec_files_by_ticker,
        sec_filings_dir     = OUTPUT_DIR_SEC_FILINGS
    )

with _TPE(max_workers=4) as _pool:
    list(_pool.map(_gen_mf, mutual_funds))


## Generate Ratings Change Report

In [ ]:
# ============================================================
# Generate Ratings Change Report
# ============================================================
# Generates a single consolidated Word doc covering all individual
# equities. Uses generate_report() for consistency with other reports.

os.makedirs(OUTPUT_DIR_PORTFOLIO_ANALYSIS, exist_ok=True)

date_str        = datetime.now().strftime('%Y-%m-%d')
output_filename = f'Ratings Change Report {date_str}.docx'
output_path     = os.path.join(OUTPUT_DIR_PORTFOLIO_ANALYSIS, output_filename)

if os.path.exists(output_path):
    pass
else:
    doc = Document()
    doc.add_heading('Ratings Change Report', 0)
    doc.add_paragraph(f'Grok Model Generated: {date_str}')
    doc.add_paragraph()

    total_in = total_out = 0

    for i, equity in enumerate(individual_equities, 1):
        prompt = f'For the equity {equity} {PROMPT_RATINGS_CHANGE_TEMPLATE}'
        doc.add_heading(equity, level=1)
        try:
            t0 = time.time()
            generated_text, llm_usage = client.chat(prompt, model=MODEL)
            elapsed = time.time() - t0
            total_in  += llm_usage.get('input_tokens', 0)
            total_out += llm_usage.get('output_tokens', 0)
            add_markdown_to_word(doc, generated_text)
        except Exception as e:
            doc.add_paragraph(f'Error processing {equity}: {e}')
        doc.add_paragraph()

    doc.add_heading('Prompt Template Used:', level=2)
    doc.add_paragraph(PROMPT_RATINGS_CHANGE_TEMPLATE)

    doc.save(output_path)


## End